**Trabalho 2 IA - JARVIS**

- Funcionalidade 3.4
- Melhorias de aprendizado
- Avaliação e análise de erros


In [1]:
# INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS

!pip install sentence-transformers faiss-cpu openai

import os
import re
import json
import unicodedata
import numpy as np
import faiss
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from openai import OpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 34.7 MB/s eta 0:00:00


In [2]:
# CRIAÇÃO DA PASTAS

os.makedirs("data", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("avaliacao", exist_ok=True)

print("Pastas criadas")

Pastas criadas


**Observação sobre os arquivos TXT**

Após executar a célula de criação das pastas, inserir os arquivos `.txt` dos materiais acadêmicos dentro da pasta `data`. Somente depois disso executar as demais células.

In [3]:
# CRIAÇÃO DOS ARQUIVOS BASE

agenda_inicial = [
    {
        "data": "2026-05-18",
        "hora": "18:30",
        "tipo": "aula",
        "disciplina": "Inteligência Artificial",
        "descricao": "Aula sobre RAG e embeddings"
    },
    {
        "data": "2026-05-26",
        "hora": "20:30",
        "tipo": "prova",
        "disciplina": "Computação Distribuída",
        "descricao": "Prova sobre protocolos de comunicação e escalabilidade"
    }
]

tarefas_iniciais = [
    {
        "id": 1,
        "descricao": "Estudar embeddings",
        "disciplina": "Inteligência Artificial",
        "prazo": "2026-05-16",
        "concluida": False
    },
    {
        "id": 2,
        "descricao": "Revisar slides de escalabilidade",
        "disciplina": "Computação Distribuída",
        "prazo": "2026-05-23",
        "concluida": False
    }
]

if not os.path.exists("agenda.json"):
    with open("agenda.json", "w", encoding="utf-8") as f:
        json.dump(agenda_inicial, f, ensure_ascii=False, indent=4)

if not os.path.exists("tarefas.json"):
    with open("tarefas.json", "w", encoding="utf-8") as f:
        json.dump(tarefas_iniciais, f, ensure_ascii=False, indent=4)

if not os.path.exists("logs/tool_calls.json"):
    with open("logs/tool_calls.json", "w", encoding="utf-8") as f:
        json.dump([], f, ensure_ascii=False, indent=4)

print("Arquivos base prontos")

Arquivos base prontos


In [4]:
# FUNÇÕES AUXILIARES UTILIZADAS PELA AGENDA ACADÊMICA

def carregar_json(caminho, valor_padrao):
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return valor_padrao
    except Exception as e:
        print("Erro ao carregar JSON:", e)
        return valor_padrao


def salvar_json(caminho, dados):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=4)


def formatar_data(data_iso):
    return datetime.strptime(data_iso, "%Y-%m-%d").strftime("%d/%m/%Y")


def interpretar_data(pergunta_usuario):
    pergunta = pergunta_usuario.lower()
    hoje = datetime.now()

    if "hoje" in pergunta:
        return hoje.strftime("%Y-%m-%d")

    if "amanhã" in pergunta or "amanha" in pergunta:
        return (hoje + timedelta(days=1)).strftime("%Y-%m-%d")

    padrao = re.search(r"(\d{2})/(\d{2})/(\d{4})", pergunta)
    if padrao:
        dia, mes, ano = padrao.groups()
        return f"{ano}-{mes}-{dia}"

    return None


def pergunta_pede_semana(pergunta_usuario):
    pergunta = pergunta_usuario.lower()
    return "semana" in pergunta or "esta semana" in pergunta

In [5]:
# FUNÇÃO: AGENDA

def consultar_agenda(data_consulta=None, semana=False):

    try:
        agenda = carregar_json("agenda.json", [])

        if not agenda:
            return "Nenhum evento cadastrado"

        if semana:
            hoje = datetime.now()
            inicio = hoje - timedelta(days=hoje.weekday())
            fim = inicio + timedelta(days=6)

            eventos = []

            for evento in agenda:
                data_evento = datetime.strptime(evento["data"], "%Y-%m-%d")

                if inicio.date() <= data_evento.date() <= fim.date():
                    eventos.append(evento)

            if not eventos:
                return "Nenhum evento encontrado para esta semana"

        else:
            if data_consulta is None:
                data_consulta = datetime.now().strftime("%Y-%m-%d")

            eventos = [
                evento for evento in agenda
                if evento["data"] == data_consulta
            ]

            if not eventos:
                return f"Nenhum evento encontrado para {formatar_data(data_consulta)}."

        resposta = "Eventos encontrados:\n\n"

        for evento in eventos:
            resposta += (
                f"Data: {formatar_data(evento['data'])}\n"
                f"Horário: {evento['hora']}\n"
                f"Tipo: {evento['tipo']}\n"
                f"Disciplina: {evento['disciplina']}\n"
                f"Descrição: {evento['descricao']}\n\n"
            )

        return resposta.strip()

    except ValueError:
        return "Erro: a data deve estar no formato AAAA-MM-DD"

    except KeyError as e:
        return f"Erro: campo ausente no evento da agenda: {str(e)}"

    except Exception as e:
        return f"Erro ao consultar agenda: {str(e)}"

In [6]:
# FUNÇÕES: TAREFAS

def listar_tarefas():

    try:
        tarefas = carregar_json("tarefas.json", [])

        if not tarefas:
            return "Nenhuma tarefa cadastrada"

        resposta = ""

        for tarefa in tarefas:

            status = "Concluída" if tarefa["concluida"] else "Pendente"

            prazo_formatado = datetime.strptime(
                tarefa["prazo"],
                "%Y-%m-%d"
            ).strftime("%d/%m/%Y")

            resposta += (
                f"Tarefa {tarefa['id']}\n"
                f"Descrição: {tarefa['descricao']}\n"
                f"Disciplina: {tarefa['disciplina']}\n"
                f"Prazo: {prazo_formatado}\n"
                f"Status: {status}\n\n"
            )

        return resposta

    except Exception as e:
        return f"Erro ao listar tarefas: {str(e)}"


def adicionar_tarefa(descricao, disciplina, prazo):

    try:
        tarefas = carregar_json("tarefas.json", [])

        novo_id = max(
            [tarefa["id"] for tarefa in tarefas],
            default=0
        ) + 1

        nova_tarefa = {
            "id": novo_id,
            "descricao": descricao,
            "disciplina": disciplina,
            "prazo": prazo,
            "concluida": False
        }

        tarefas.append(nova_tarefa)

        salvar_json("tarefas.json", tarefas)

        return f"Tarefa adicionada com ID {novo_id}"

    except Exception as e:
        return f"Erro ao adicionar tarefa: {str(e)}"


def concluir_tarefa(id_tarefa):

    try:
        tarefas = carregar_json("tarefas.json", [])

        for tarefa in tarefas:
            if tarefa["id"] == int(id_tarefa):
                tarefa["concluida"] = True
                salvar_json("tarefas.json", tarefas)
                return f"Tarefa {id_tarefa} marcada como concluída"

        return f"Tarefa {id_tarefa} não encontrada"

    except Exception as e:
        return f"Erro ao concluir tarefa: {str(e)}"

print(listar_tarefas())

Tarefa 1
Descrição: Estudar embeddings
Disciplina: Inteligência Artificial
Prazo: 16/05/2026
Status: Pendente

Tarefa 2
Descrição: Revisar slides de escalabilidade
Disciplina: Computação Distribuída
Prazo: 23/05/2026
Status: Pendente




In [7]:
# LEITURA E TRATAMENTO DOS ARQUIVOS

documentos = []

def limpar_texto(texto):
    texto = texto.replace("\ufeff", "")
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.replace("\u00a0", " ")
    texto = texto.replace("\t", " ")
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ ]+", " ", texto)
    texto = re.sub(r"\s+([.,;:!?])", r"\1", texto)
    return texto.strip()


arquivos_txt = [
    arquivo for arquivo in os.listdir("data")
    if arquivo.lower().endswith(".txt")
]

if not arquivos_txt:
    print("Nenhum arquivo encontrado na pasta data")
else:
    for arquivo in arquivos_txt:
        caminho = os.path.join("data", arquivo)

        with open(caminho, "r", encoding="utf-8") as f:
            texto = f.read()

        documentos.append({
            "arquivo": arquivo,
            "texto": limpar_texto(texto)
        })

    print("Documentos carregados:", len(documentos))

Documentos carregados: 10


In [8]:
# DIVISÃO DOS TEXTOS EM CHUNKS

def dividir_em_chunks(texto, tamanho_maximo=1000):
    frases = texto.split(". ")
    chunks = []
    chunk_atual = ""

    for frase in frases:
        frase = frase.strip()

        if not frase:
            continue

        if not frase.endswith("."):
            frase += "."

        if len(chunk_atual) + len(frase) <= tamanho_maximo:
            chunk_atual += " " + frase
        else:
            if chunk_atual.strip():
                chunks.append(chunk_atual.strip())
            chunk_atual = frase

    if chunk_atual.strip():
        chunks.append(chunk_atual.strip())

    return chunks


chunks = []

for documento in documentos:
    partes = dividir_em_chunks(documento["texto"])

    for i, parte in enumerate(partes):
        chunks.append({
            "arquivo": documento["arquivo"],
            "chunk_id": i,
            "texto": parte
        })

print("Total de chunks:", len(chunks))

Total de chunks: 77


In [9]:
# CRIAÇÃO DOS EMBEDDINGS E ÍNDICE FAISS

if not chunks:
    print("Nenhum chunk encontrado. Confira se os arquivos TXT foram inseridos na pasta data")
else:
    modelo_embeddings = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    textos_chunks = [chunk["texto"] for chunk in chunks]

    embeddings = modelo_embeddings.encode(
        textos_chunks,
        convert_to_numpy=True
    ).astype("float32")

    dimensao = embeddings.shape[1]

    indice_faiss = faiss.IndexFlatL2(dimensao)
    indice_faiss.add(embeddings)

    print("Total de vetores:", indice_faiss.ntotal)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total de vetores: 77


In [10]:
# CONEXÃO COM A API

# Por motivos de segurança, o token da API não foi incluído no repositório
# Antes de executar o sistema, substitua abaixo pelo token original

TOKEN_LLM = 'REIkURcI7rTTqsTwlJi8MrgnKFw0iqky7Ezh7hH-l-k'

client = OpenAI(
    base_url="https://llm.liaufms.org/v1/qwen2-5-14b-instruct-awq",
    api_key=TOKEN_LLM,
    timeout=60
)

def chamar_llm(prompt):

    resposta = client.chat.completions.create(
        model="Qwen/Qwen2.5-14B-Instruct-AWQ",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return resposta.choices[0].message.content.strip()

In [11]:
print(chamar_llm("Olá"))

AuthenticationError: Error code: 401 - {'error': 'http_error', 'message': 'Invalid API token', 'request_id': ''}

In [12]:
# FUNÇÃO: RESPONDER COM RAG

def recuperar_trechos(pergunta, top_k=3):
    if not chunks:
        return []

    embedding_pergunta = modelo_embeddings.encode(
        [pergunta],
        convert_to_numpy=True
    ).astype("float32")

    distancias, indices = indice_faiss.search(
        embedding_pergunta,
        top_k
    )

    trechos = []

    for indice in indices[0]:
        chunk = chunks[indice]
        trechos.append({
            "arquivo": chunk["arquivo"],
            "chunk_id": chunk["chunk_id"],
            "texto": chunk["texto"]
        })

    return trechos


def buscar_material_rag(pergunta, top_k=3):
    trechos = recuperar_trechos(pergunta, top_k)

    if not trechos:
        return "Nenhum trecho encontrado nos materiais"

    contexto = ""

    for trecho in trechos:
        contexto += (
            f"Fonte: {trecho['arquivo']} | Chunk: {trecho['chunk_id']}\n"
            f"{trecho['texto']}\n\n"
        )

    prompt = f'''
Responda à pergunta utilizando apenas os trechos abaixo
Se a resposta não estiver nos trechos, diga que não encontrou informação suficiente

Pergunta:
{pergunta}

Trechos:
{contexto}
'''

    return chamar_llm(prompt)

In [13]:
#LOGS

def registrar_log(ferramenta, entrada, saida):
    logs = carregar_json("logs/tool_calls.json", [])

    registro = {
        "data_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "ferramenta": ferramenta,
        "entrada": entrada,
        "saida": saida
    }

    logs.append(registro)
    salvar_json("logs/tool_calls.json", logs)


def listar_logs():
    logs = carregar_json("logs/tool_calls.json", [])

    if not logs:
        return "Nenhum log registrado"

    resposta = "Logs registrados:\n\n"

    for log in logs:
        resposta += (
            f"Data/hora: {log['data_hora']}\n"
            f"Ferramenta: {log['ferramenta']}\n"
            f"Entrada: {log['entrada']}\n"
            f"Saída: {str(log['saida'])[:300]}...\n\n"
        )

    return resposta.strip()

In [14]:
#PLANEJAMENTO DE ESTUDOS

def planejar_estudos(pergunta):
    agenda = consultar_agenda(semana=True)
    tarefas = listar_tarefas()
    trechos = recuperar_trechos(pergunta, top_k=3)

    contexto_materiais = ""
    for trecho in trechos:
        contexto_materiais += (
            f"Fonte: {trecho['arquivo']}\n"
            f"{trecho['texto']}\n\n"
        )

    prompt = f'''
Você é um assistente acadêmico

Monte um plano de estudos objetivo com base na agenda, nas tarefas e nos materiais recuperados

Pedido do usuário:
{pergunta}

Agenda:
{agenda}

Tarefas:
{tarefas}

Materiais relacionados:
{contexto_materiais}

Responda em tópicos curtos, indicando prioridades
'''

    return chamar_llm(prompt)

In [15]:
#FUNCIONALIDADES DE APRENDIZADO

def gerar_exercicios(tema, quantidade=5):
    trechos = recuperar_trechos(tema, top_k=3)

    contexto = ""
    for trecho in trechos:
        contexto += f"{trecho['texto']}\n\n"

    prompt = f'''
Crie {quantidade} exercícios sobre o tema abaixo, usando o contexto dos materiais

Tema:
{tema}

Contexto:
{contexto}

Inclua as respostas ao final
'''

    return chamar_llm(prompt)


def recomendar_revisao(tema):
    trechos = recuperar_trechos(tema, top_k=3)

    contexto = ""
    for trecho in trechos:
        contexto += f"{trecho['texto']}\n\n"

    prompt = f'''
Com base no material abaixo, recomende o que o estudante deve revisar sobre o tema

Tema:
{tema}

Material:
{contexto}

A resposta deve ser direta e organizada por prioridades
'''

    return chamar_llm(prompt)


def active_recall(tema, quantidade=3):
    trechos = recuperar_trechos(tema, top_k=3)

    contexto = ""
    for trecho in trechos:
        contexto += f"{trecho['texto']}\n\n"

    prompt_perguntas = f'''
Crie {quantidade} perguntas de active recall sobre o tema abaixo
Retorne apenas as perguntas numeradas

Tema:
{tema}

Contexto:
{contexto}
'''

    perguntas = chamar_llm(prompt_perguntas)
    print(perguntas)

    resposta_usuario = input("\nResponda às perguntas acima: ")

    prompt_avaliacao = f'''
Avalie a resposta do estudante com base no contexto

Tema:
{tema}

Contexto:
{contexto}

Perguntas:
{perguntas}

Resposta do estudante:
{resposta_usuario}

Dê um feedback curto, diga o que está correto e o que precisa melhorar
'''

    return chamar_llm(prompt_avaliacao)

In [16]:
#DECISÃO PELA LLM

def extrair_json(texto):
    try:
        return json.loads(texto)
    except:
        pass

    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass

    return {
        "ferramenta": "buscar_material_rag",
        "entrada": {"pergunta": texto}
    }


def decidir_ferramenta_llm(pergunta_usuario):
    prompt = f'''
Você é o roteador de ferramentas do JARVIS Acadêmico

Escolha a ferramenta mais adequada para responder ao usuário.

Ferramentas disponíveis:

1. consultar_agenda
Entrada: {{"data_consulta": "AAAA-MM-DD", "semana": false}}

2. listar_tarefas
Entrada: {{}}

3. adicionar_tarefa
Entrada: {{"descricao": "...", "disciplina": "...", "prazo": "AAAA-MM-DD"}}

4. concluir_tarefa
Entrada: {{"id_tarefa": 1}}

5. buscar_material_rag
Entrada: {{"pergunta": "..."}}

6. planejar_estudos
Entrada: {{"pergunta": "..."}}

7. gerar_exercicios
Entrada: {{"tema": "...", "quantidade": 5}}

8. recomendar_revisao
Entrada: {{"tema": "..."}}

9. active_recall
Entrada: {{"tema": "..."}}

Responda somente em JSON válido, neste formato:
{{"ferramenta": "nome_da_ferramenta", "entrada": {{...}}}}

Pergunta do usuário:
{pergunta_usuario}
'''

    resposta = chamar_llm(prompt)
    decisao = extrair_json(resposta)

    ferramenta = decisao.get("ferramenta", "buscar_material_rag")
    entrada = decisao.get("entrada", {})

    if ferramenta == "consultar_agenda":
        if not entrada.get("data_consulta"):
            entrada["data_consulta"] = interpretar_data(pergunta_usuario)
        entrada["semana"] = entrada.get("semana", pergunta_pede_semana(pergunta_usuario))

    if ferramenta == "buscar_material_rag":
        entrada["pergunta"] = entrada.get("pergunta", pergunta_usuario)

    if ferramenta == "planejar_estudos":
        entrada["pergunta"] = entrada.get("pergunta", pergunta_usuario)

    if ferramenta == "active_recall":
        entrada["tema"] = entrada.get("tema", pergunta_usuario)

    return {
        "ferramenta": ferramenta,
        "entrada": entrada
    }

In [17]:
#EXECUTOR CENTRAL DAS FERRAMENTAS

def executar_ferramenta(nome_ferramenta, entrada=None):
    if entrada is None:
        entrada = {}

    try:
        if nome_ferramenta == "consultar_agenda":
            saida = consultar_agenda(**entrada)

        elif nome_ferramenta == "listar_tarefas":
            saida = listar_tarefas()

        elif nome_ferramenta == "adicionar_tarefa":
            saida = adicionar_tarefa(**entrada)

        elif nome_ferramenta == "concluir_tarefa":
            saida = concluir_tarefa(**entrada)

        elif nome_ferramenta == "buscar_material_rag":
            saida = buscar_material_rag(**entrada)

        elif nome_ferramenta == "planejar_estudos":
            saida = planejar_estudos(**entrada)

        elif nome_ferramenta == "gerar_exercicios":
            saida = gerar_exercicios(**entrada)

        elif nome_ferramenta == "recomendar_revisao":
            saida = recomendar_revisao(**entrada)

        elif nome_ferramenta == "active_recall":
            saida = active_recall(**entrada)

        else:
            saida = f"Ferramenta desconhecida: {nome_ferramenta}"

        registrar_log(nome_ferramenta, entrada, saida)
        return saida

    except Exception as e:
        erro = f"Erro ao executar ferramenta: {str(e)}"
        registrar_log(nome_ferramenta, entrada, erro)
        return erro

In [18]:
#FUNÇÃO JARVIS

def jarvis(pergunta_usuario):
    decisao = decidir_ferramenta_llm(pergunta_usuario)

    return executar_ferramenta(
        decisao["ferramenta"],
        decisao["entrada"]
    )

In [19]:
#AVALIAÇÃO DO SISTEMA

perguntas_avaliacao = [
    "Explique o que é KNN",
    "O que são embeddings?",
    "Qual a relação entre embeddings e RAG?",
    "O que são redes neurais?",
    "Explique Deep Learning",
    "O que é Processamento de Linguagem Natural?",
    "O que é clustering?",
    "Explique overfitting e underfitting",
    "O que tenho na agenda esta semana?",
    "Quais tarefas estão cadastradas?"
]


def avaliar_sistema(perguntas, classificacoes=None):
    resultados = []

    if classificacoes is None:
        classificacoes = ["pendente"] * len(perguntas)

    for pergunta, classificacao in zip(perguntas, classificacoes):
        trechos = recuperar_trechos(pergunta, top_k=3)
        resposta = jarvis(pergunta)

        resultado = {
            "pergunta": pergunta,
            "documentos_recuperados": [
                {
                    "arquivo": trecho["arquivo"],
                    "chunk_id": trecho["chunk_id"]
                }
                for trecho in trechos
            ],
            "resposta": resposta,
            "classificacao": classificacao
        }

        resultados.append(resultado)

    salvar_json("avaliacao/avaliacao_sistema.json", resultados)

    return resultados

In [20]:
#ANÁLISE DE ERROS

analise_erros = [
    {
        "falha": "Recuperação de trecho pouco específico",
        "tipo": "recuperação",
        "causa": "A pergunta pode ser genérica ou existir conteúdo semelhante em vários documentos",
        "possivel_solucao": "Melhorar o chunking, aumentar o top_k ou tornar a pergunta mais específica"
    },
    {
        "falha": "Resposta incompleta da LLM",
        "tipo": "geração",
        "causa": "A LLM pode resumir demais ou não utilizar todo o contexto recuperado",
        "possivel_solucao": "Ajustar o prompt para exigir resposta mais detalhada e baseada nas fontes"
    },
    {
        "falha": "Ambiguidade na escolha da ferramenta",
        "tipo": "ambiguidade",
        "causa": "Perguntas muito abertas podem ser interpretadas como RAG ou planejamento",
        "possivel_solucao": "Melhorar o prompt do roteador e incluir exemplos de decisão para cada ferramenta"
    }
]

salvar_json("avaliacao/analise_erros.json", analise_erros)

print("Arquivo de análise de erros criado")

Arquivo de análise de erros criado


In [21]:
# TESTES BÁSICOS

print(listar_tarefas())
print("\n" + "=" * 80)
print(consultar_agenda())
print("\n" + "=" * 80)
print(recuperar_trechos("embeddings", top_k=2))
print("\n" + "=" * 80)
print(listar_logs())

Tarefa 1
Descrição: Estudar embeddings
Disciplina: Inteligência Artificial
Prazo: 16/05/2026
Status: Pendente

Tarefa 2
Descrição: Revisar slides de escalabilidade
Disciplina: Computação Distribuída
Prazo: 23/05/2026
Status: Pendente



Nenhum evento encontrado para 14/06/2026.

[{'arquivo': 'Embeddings.txt', 'chunk_id': 6, 'texto': 'Esses modelos conseguem representar palavras considerando o contexto em que aparecem, produzindo embeddings muito mais precisos e eficientes.\nAtualmente, grandes modelos de linguagem utilizam embeddings como parte fundamental do processamento textual. Sistemas modernos conseguem transformar perguntas, documentos e conversas inteiras em representações vetoriais utilizadas em tarefas de recuperação semântica, classificação e geração de texto.\nApesar das vantagens, os embeddings também apresentam limitações importantes. O treinamento desses modelos exige grande volume de dados e capacidade computacional elevada. Além disso, embeddings treinados em bases ina

In [22]:
#TESTES

print("\n" + "=" * 80)
print(jarvis("Explique o que é KNN"))

print("\n" + "=" * 80)

print("\n" + "=" * 80)
print(jarvis("Quais tarefas estão cadastradas?"))

print("\n" + "=" * 80)

print("\n" + "=" * 80)
print(jarvis("O que tenho na agenda esta semana?"))

print("\n" + "=" * 80)

print("\n" + "=" * 80)
print(jarvis("Monte um plano de estudos para revisar embeddings e RAG"))
print("\n" + "=" * 80)

AuthenticationError: Error code: 401 - {'error': 'http_error', 'message': 'Invalid API token', 'request_id': ''}